# Pipeline Test Notebook
Test the `utils.py` S3 helpers and `pipeline.py` parse + upsert functions end-to-end:
1. List available dumps in S3
2. Download one dump locally
3. Parse it with `parse_dump()`
4. Inspect the aggregates
5. Clean up the temp file
6. **Upsert first 3 dumps into Postgres** via `upsert_aggregates()`
7. Compute and verify `artist_daily_stats` / `track_daily_stats` via `compute_daily_stats()`

In [1]:
import importlib
import sys
from pathlib import Path

# Add parent dir to path so we can import local modules
sys.path.insert(0, str(Path.cwd().parent))

# Force-reload to pick up any changes since the kernel started
import pipeline, utils
importlib.reload(utils)
importlib.reload(pipeline)

from utils import get_s3_client, list_s3_artifacts, download_s3_dump, connect_postgres, load_db_credentials
from pipeline import parse_dump, upsert_aggregates, compute_daily_stats

print("Imports OK")

Imports OK


In [2]:
import os
from dotenv import load_dotenv

# Load AWS creds from the project .env
env_path = Path.cwd().parent.parent / ".env"
print(f"Loading env from: {env_path}  (exists={env_path.exists()})")
load_dotenv(dotenv_path=env_path, override=True)

# Verify AWS creds are set
for k in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN", "AWS_DEFAULT_REGION"]:
    v = os.getenv(k)
    print(f"  {k} = {'set (' + v[:8] + '...)' if v else 'NOT SET'}")

Loading env from: c:\Users\Natha\source\repos\Classes\DE_300\project\.env  (exists=True)
  AWS_ACCESS_KEY_ID = set (ASIAYAAO...)
  AWS_SECRET_ACCESS_KEY = set (yS4h1wNG...)
  AWS_SESSION_TOKEN = set (IQoJb3Jp...)
  AWS_DEFAULT_REGION = set (us-east-...)


In [3]:
# Force a fresh boto3 session that re-reads env vars
import boto3
boto3.DEFAULT_SESSION = None
session = boto3.Session()

sts = session.client("sts")
identity = sts.get_caller_identity()
print(f"Authenticated as: {identity['Arn']}")
print(f"Account: {identity['Account']}")

Authenticated as: arn:aws:sts::549787090008:assumed-role/AWSReservedSSO_mse-tl-dataeng300-EMR_da0cc2e9e5742c69/alt5629
Account: 549787090008


In [5]:
BUCKET = "stall-munezero-final-project"

# Use a fresh client from the reset session
s3 = session.client("s3")

# Test list_s3_artifacts (pass explicit client to bypass get_s3_client's cached session)
keys = list_s3_artifacts(BUCKET, "listenbrainz/incremental/", s3_client=s3)
print(f"✓ list_s3_artifacts found {len(keys)} keys:")
for k in keys:
    print(f"  {k}")

✓ list_s3_artifacts found 41 keys:
  listenbrainz/incremental/listenbrainz-listens-dump-2400-20260118-000003-incremental.tar.zst
  listenbrainz/incremental/listenbrainz-listens-dump-2401-20260119-000003-incremental.tar.zst
  listenbrainz/incremental/listenbrainz-listens-dump-2402-20260120-000003-incremental.tar.zst
  listenbrainz/incremental/listenbrainz-listens-dump-2403-20260120-200718-incremental.tar.zst
  listenbrainz/incremental/listenbrainz-listens-dump-2404-20260121-000003-incremental.tar.zst
  listenbrainz/incremental/listenbrainz-listens-dump-2405-20260122-000003-incremental.tar.zst
  listenbrainz/incremental/listenbrainz-listens-dump-2406-20260123-000003-incremental.tar.zst
  listenbrainz/incremental/listenbrainz-listens-dump-2407-20260124-000003-incremental.tar.zst
  listenbrainz/incremental/listenbrainz-listens-dump-2408-20260125-000003-incremental.tar.zst
  listenbrainz/incremental/listenbrainz-listens-dump-2409-20260126-000003-incremental.tar.zst
  listenbrainz/incrementa

In [26]:
# Download a dump from S3 and parse it
# Pick the first key from the list
dump_key = keys[0]
print(f"Downloading: {dump_key}")

local_path = download_s3_dump(BUCKET, dump_key, local_dir=Path("./tmp_test"), s3_client=s3)
print(f"Local path: {local_path}")
print(f"Size: {local_path.stat().st_size / 1e6:.1f} MB")

Downloading: listenbrainz/incremental/listenbrainz-listens-dump-2400-20260118-000003-incremental.tar.zst
Done (184.4 MB)
Local path: tmp_test\listenbrainz-listens-dump-2400-20260118-000003-incremental.tar.zst
Size: 184.4 MB


In [27]:
# Parse the downloaded S3 dump (cap at 50k lines for quick test)
track_daily, artist_daily, track_info, artist_info, summary = parse_dump(local_path, max_lines=50_000)

print("Parse summary:")
for k, v in summary.items():
    print(f"  {k}: {v:,}")

Parsing listens: 50000it [00:00, 73145.46it/s]

Parse summary:
  lines_parsed: 50,001
  bad_json: 0
  missing_timestamp: 0
  missing_artist_mbid: 0
  unique_track_day_keys: 43,843
  unique_artist_day_keys: 1,265
  unique_tracks: 28,150
  unique_artists: 1,160


In [28]:
# Cleanup temp file
local_path.unlink()
print(f"Deleted: {local_path}")

# Remove temp dir if empty
import shutil
tmp_dir = Path("./tmp_test")
if tmp_dir.exists() and not any(tmp_dir.iterdir()):
    tmp_dir.rmdir()
    print(f"Removed empty dir: {tmp_dir}")

Deleted: tmp_test\listenbrainz-listens-dump-2400-20260118-000003-incremental.tar.zst
Removed empty dir: tmp_test


## 6) Upsert first 3 dumps into Postgres
Downloads each dump from S3, parses it fully, upserts aggregates, then cleans up the temp file before moving to the next.

In [29]:
# Pick the first 3 dumps from S3
dumps_to_ingest = keys[:3]
print(f"Will ingest {len(dumps_to_ingest)} dumps:")
for k in dumps_to_ingest:
    print(f"  {k}")

Will ingest 3 dumps:
  listenbrainz/incremental/listenbrainz-listens-dump-2400-20260118-000003-incremental.tar.zst
  listenbrainz/incremental/listenbrainz-listens-dump-2401-20260119-000003-incremental.tar.zst
  listenbrainz/incremental/listenbrainz-listens-dump-2402-20260120-000003-incremental.tar.zst


In [40]:
import time

tmp_dir = Path("./tmp_test")

for i, dump_key in enumerate(dumps_to_ingest, 1):
    print(f"\n{'='*60}")
    print(f"[{i}/{len(dumps_to_ingest)}] {dump_key}")
    print(f"{'='*60}")

    # Download
    t0 = time.perf_counter()
    local_path = download_s3_dump(BUCKET, dump_key, local_dir=tmp_dir, s3_client=s3)

    # Parse (full — no line limit)
    track_daily, artist_daily, track_info, artist_info, summary = parse_dump(local_path)
    print("Parse summary:")
    for k, v in summary.items():
        print(f"  {k}: {v:,}")

    # Upsert
    conn = connect_postgres()
    upsert_aggregates(conn, track_daily, artist_daily, track_info, artist_info, dump_path=dump_key)
    conn.close()

    # Cleanup
    local_path.unlink()
    elapsed = time.perf_counter() - t0
    print(f"Done in {elapsed:.1f}s — temp file deleted.")

# Remove temp dir if empty
if tmp_dir.exists() and not any(tmp_dir.iterdir()):
    tmp_dir.rmdir()
    print(f"\nRemoved empty dir: {tmp_dir}")

print("\nAll 3 dumps ingested.")


[1/3] listenbrainz/incremental/listenbrainz-listens-dump-2400-20260118-000003-incremental.tar.zst
Done (184.4 MB)


Parsing listens: 3731749it [00:45, 81593.54it/s] 


Parse summary:
  lines_parsed: 3,731,749
  bad_json: 0
  missing_timestamp: 0
  missing_artist_mbid: 0
  unique_track_day_keys: 2,933,549
  unique_artist_day_keys: 14,636
  unique_tracks: 616,535
  unique_artists: 14,070
  Upserting artist_info (14,070 rows)... done (6.1s)
  Upserting track_info (616,535 rows)... done (273.3s)
  Upserting track_daily_listens (2,933,549 rows)... done (1344.0s)
  Upserting artist_daily_listens (14,636 rows)... done (7.2s)
Upserts complete.
Done in 1683.4s — temp file deleted.

[2/3] listenbrainz/incremental/listenbrainz-listens-dump-2401-20260119-000003-incremental.tar.zst
Done (118.6 MB)


Parsing listens: 2320214it [00:30, 76549.43it/s] 


Parse summary:
  lines_parsed: 2,320,214
  bad_json: 0
  missing_timestamp: 0
  missing_artist_mbid: 0
  unique_track_day_keys: 1,692,846
  unique_artist_day_keys: 14,683
  unique_tracks: 512,115
  unique_artists: 14,055
  Upserting artist_info (14,055 rows)... done (9.0s)
  Upserting track_info (512,115 rows)... done (263.9s)
  Upserting track_daily_listens (1,692,846 rows)... done (805.9s)
  Upserting artist_daily_listens (14,683 rows)... done (7.0s)
Upserts complete.
Done in 1121.7s — temp file deleted.

[3/3] listenbrainz/incremental/listenbrainz-listens-dump-2402-20260120-000003-incremental.tar.zst
Done (154.4 MB)


Parsing listens: 3087943it [00:38, 79879.21it/s] 


Parse summary:
  lines_parsed: 3,087,943
  bad_json: 0
  missing_timestamp: 0
  missing_artist_mbid: 0
  unique_track_day_keys: 2,327,516
  unique_artist_day_keys: 16,653
  unique_tracks: 568,280
  unique_artists: 16,095
  Upserting artist_info (16,095 rows)... done (7.3s)
  Upserting track_info (568,280 rows)... done (261.9s)
  Upserting track_daily_listens (2,327,516 rows)... done (977.3s)
  Upserting artist_daily_listens (16,653 rows)... done (8.0s)
Upserts complete.
Done in 1302.6s — temp file deleted.

Removed empty dir: tmp_test

All 3 dumps ingested.


In [41]:
# Verify: check row counts and ingestion_state
conn = connect_postgres()
with conn.cursor() as cur:
    for table in ["artist_info", "track_info", "artist_daily_listens", "track_daily_listens"]:
        cur.execute(f"SELECT COUNT(*) FROM {table};")  # noqa: S608 — table names are hardcoded
        print(f"  {table}: {cur.fetchone()[0]:,} rows")

    cur.execute("SELECT last_dump_id, last_dump_path, loaded_at FROM ingestion_state WHERE id = 1;")
    row = cur.fetchone()
    print(f"\ningestion_state:")
    print(f"  last_dump_id:   {row[0]}")
    print(f"  last_dump_path: {row[1]}")
    print(f"  loaded_at:      {row[2]}")
conn.close()

  artist_info: 29,552 rows
  track_info: 1,285,463 rows
  artist_daily_listens: 45,261 rows
  track_daily_listens: 6,815,895 rows

ingestion_state:
  last_dump_id:   2402
  last_dump_path: listenbrainz/incremental/listenbrainz-listens-dump-2402-20260120-000003-incremental.tar.zst
  loaded_at:      2026-03-06 05:07:56.002988+00:00


In [4]:
# Build daily stats tables from the ingested listens tables
import importlib
import os
from pathlib import Path
import pipeline

# Ensure Windows Spark can start (HADOOP_HOME/hadoop.home.dir must exist before JVM boot).
if os.name == "nt":
    hadoop_home = Path.home() / ".hadoop"
    (hadoop_home / "bin").mkdir(parents=True, exist_ok=True)
    os.environ["HADOOP_HOME"] = str(hadoop_home)
    os.environ["hadoop.home.dir"] = str(hadoop_home)
    print(f"HADOOP_HOME set to: {hadoop_home}")

# Reload pipeline so notebook picks up latest compute_daily_stats changes.
importlib.reload(pipeline)

creds = load_db_credentials()
print("Computing daily stats (artist + track)...")
pipeline.compute_daily_stats(creds)

# Verify stats tables now contain rows
conn = connect_postgres()
with conn.cursor() as cur:
    for table in ["artist_daily_stats", "track_daily_stats"]:
        cur.execute(f"SELECT COUNT(*) FROM {table};")  # noqa: S608 — table names are hardcoded
        print(f"  {table}: {cur.fetchone()[0]:,} rows")
conn.close()

HADOOP_HOME set to: C:\Users\Natha\.hadoop
Computing daily stats (artist + track)...
Loaded 45,261 rows from artist_daily_listens
Wrote 45,261 rows to artist_daily_stats
Loaded 6,815,895 rows from track_daily_listens
Wrote 6,815,895 rows to track_daily_stats
Spark session stopped.
  artist_daily_stats: 45,261 rows
  track_daily_stats: 6,815,895 rows
